In [2]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

db = duckdb.connect(
    "C:/Users/Decss/Desktop/SQL-övning/Data/retail.db",
    read_only=True
)
db.sql("SET search_path TO mart;")
db.sql("""SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'mart'""")

┌─────────────────┐
│   table_name    │
│     varchar     │
├─────────────────┤
│ customerdim     │
│ productdim      │
│ retail_enriched │
└─────────────────┘

# Revenue Overview

Uppdrag att ta fram data på försäljningen

* Hur mycket säljer vi för?
* Hur ser intäkterna ut över tid?
* Finns det säsongsvariation i intäkterna?


Notering:
Inga större filter tillagda för att visa totala omsättning.

In [3]:
# Totala intäkter inkluderar alla positiva transaktioner. 
df_dataset_revenue = db.sql("SELECT ROUND(SUM(total_price)) as Total_Dataset_Revenue FROM retail_enriched WHERE total_price > 0 ").df()
print(df_dataset_revenue)

   Total_Dataset_Revenue
0             10642111.0


In [4]:
df_revenue = db.sql("""SELECT
       YEAR(invoicedate) as Year,
       ROUND(SUM(total_price), 2) as revenue
       FROM retail_enriched
       WHERE total_price > 0 
       AND is_product = 1
       GROUP by 1
       ORDER BY 1
       """).df()
print(df_revenue)

   Year     revenue
0  2010   775286.18
1  2011  9480687.56


In [5]:
px.bar(
    df_revenue,
    x='Year',
    y='revenue',
    title='Revenue per Year'
)

På grund av den begränsade tidsperioden som datan sträcker sig så är det inte möjligt att säga något om utveckling YoY. Då datan sträcker sig från 2010-12 till 2011-12 så lägger vi fokus på den månatliga och veckovisa utvecklingen istället.

In [6]:
df_rev_month = db.sql("""
       SELECT
       DATE_TRUNC('Month', invoicedate) as month,
       SUM(total_price) as revenue
       FROM retail_enriched
       WHERE total_price > 0 
       GROUP BY 1
       ORDER BY 1
       """).df()
print(df_rev_month)

        month      revenue
0  2010-12-01   821452.730
1  2011-01-01   689811.610
2  2011-02-01   522545.560
3  2011-03-01   716215.260
4  2011-04-01   536968.491
5  2011-05-01   769296.610
6  2011-06-01   760547.010
7  2011-07-01   718076.121
8  2011-08-01   757841.380
9  2011-09-01  1056435.192
10 2011-10-01  1151263.730
11 2011-11-01  1503866.780
12 2011-12-01   637790.330


Högsäsong inom retail tenderar att vara runt november och december. Försäljningen är tydligt stark där jämfört med första 2-3 kvartal av året. I och med att vi endast har nio dagar av december 2011-data så är en jämförelse mot hela december föregående år svårt. 

In [7]:
rev_month_fig = px.line(
    df_rev_month,
    x='month',
    y='revenue',
    title='Revenue per Month'    
)

rev_month_fig.write_html(
    "C:/Users/Decss/Desktop/SQL-övning/Analysis/HTML/monthly_revenue.html",
    include_plotlyjs='cdn'
    )
rev_month_fig.show()


Försäljningen trendar svagt uppåt under årets första 8 månader och accellererar sedan i september för att sedan göra ett riktigt starkt november.

Bryter vi ned det till veckovis data så kan vi få en eller två jämförbara veckor på årsbasis.

In [8]:
df_rev_week = db.sql("""
       SELECT
       DATE_TRUNC('week', invoicedate) as week_start,
       SUM(total_price) as revenue
       FROM retail_enriched
       WHERE total_price > 0 
       GROUP BY 1
       ORDER BY 1
       """).df()
print(df_rev_week)

   week_start     revenue
0  2010-11-29  184669.470
1  2010-12-06  329108.220
2  2010-12-13  215357.040
3  2010-12-20   92318.000
4  2011-01-03  133429.720
5  2011-01-10  192991.950
6  2011-01-17  215227.010
7  2011-01-24  124534.220
8  2011-01-31  125847.360
9  2011-02-07  106893.220
10 2011-02-14  143354.210
11 2011-02-21  148404.190
12 2011-02-28  132503.550
13 2011-03-07  135076.460
14 2011-03-14  161082.620
15 2011-03-21  149292.660
16 2011-03-28  192398.080
17 2011-04-04  134211.440
18 2011-04-11  149357.401
19 2011-04-18  141530.360
20 2011-04-25   86364.580
21 2011-05-02  146706.540
22 2011-05-09  211708.330
23 2011-05-16  215384.190
24 2011-05-23  165886.830
25 2011-05-30  118357.150
26 2011-06-06  219836.760
27 2011-06-13  189877.880
28 2011-06-20  135745.710
29 2011-06-27  138987.060
30 2011-07-04  177154.310
31 2011-07-11  136911.470
32 2011-07-18  202130.170
33 2011-07-25  182275.231
34 2011-08-01  170460.170
35 2011-08-08  190121.410
36 2011-08-15  175570.680
37 2011-08-2

I den veckovisa försäljningen så kan vi jämföra de två första veckorna med de två sista veckorna och se att försäljningen har ökat betydligt senaste året på veckobasis. Speglar denna förändring även tidigare under 2010 så har 2011 varit ett riktigt bra år. 

Vi skulle senare kunna kika på hur verksamheten har förändrats och vad skillnaderna i kunder och utbud är mellan dessa två perioder.

Nedan är graf på veckovis försäljning.

In [9]:
fig_rev_week = px.line(
    df_rev_week,
    x='week_start',
    y='revenue',
    title='Revenue per Week'
)
fig_rev_week.write_html("C:/Users/Decss/Desktop/SQL-övning/Analysis/HTML/weekly_revenue.html",
    include_plotlyjs='cdn')
fig_rev_week.show()

In [10]:
df_rev_week_growth = db.sql("""
                            WITH a AS (
                            SELECT
                            DATE_TRUNC('week', invoicedate) as week,
                            SUM(total_price) as revenue
                            FROM retail_enriched
                            WHERE total_price > 0
                            GROUP BY 1
                            )

                            SELECT
                            week,
                            revenue,
                            LAG(revenue) OVER(ORDER BY week) as lag_revenue,
                            (ROUND((revenue - LAG(revenue) OVER(ORDER BY week)) / LAG(revenue) OVER(ORDER BY week), 3) * 100) as weekly_growth
                            FROM a
                            ORDER BY week 
                            
                            """).df()
print(df_rev_week_growth)

         week     revenue  lag_revenue  weekly_growth
0  2010-11-29  184669.470          NaN            NaN
1  2010-12-06  329108.220   184669.470           78.2
2  2010-12-13  215357.040   329108.220          -34.6
3  2010-12-20   92318.000   215357.040          -57.1
4  2011-01-03  133429.720    92318.000           44.5
5  2011-01-10  192991.950   133429.720           44.6
6  2011-01-17  215227.010   192991.950           11.5
7  2011-01-24  124534.220   215227.010          -42.1
8  2011-01-31  125847.360   124534.220            1.1
9  2011-02-07  106893.220   125847.360          -15.1
10 2011-02-14  143354.210   106893.220           34.1
11 2011-02-21  148404.190   143354.210            3.5
12 2011-02-28  132503.550   148404.190          -10.7
13 2011-03-07  135076.460   132503.550            1.9
14 2011-03-14  161082.620   135076.460           19.3
15 2011-03-21  149292.660   161082.620           -7.3
16 2011-03-28  192398.080   149292.660           28.9
17 2011-04-04  134211.440   

In [11]:
fig_wow_rev_growth = px.line(
    df_rev_week_growth,
    x='week',
    y='weekly_growth',
    title='Revenue week over week growth'
)
fig_wow_rev_growth.write_html("C:/Users/Decss/Desktop/SQL-övning/Analysis/HTML/wow_revenue_growth.html",
    include_plotlyjs='cdn')
fig_wow_rev_growth.show()

Försäljningen har haft en tydlig trend uppåt sedan början på 2011.
För att komplettera detta så tittar vi även på hur orderläggning har utvecklats över tid.

In [12]:
df_order_month = db.sql("""
                        SELECT
                        DATE_TRUNC('Month', invoicedate) as month,
                        COUNT(DISTINCT invoiceno) as orders
                        FROM retail_enriched
                        WHERE is_credit_invoice = 0
                        GROUP BY 1
                        ORDER BY 1
                        """).df()
print(df_order_month)

        month  orders
0  2010-12-01    1699
1  2011-01-01    1216
2  2011-02-01    1174
3  2011-03-01    1665
4  2011-04-01    1504
5  2011-05-01    1848
6  2011-06-01    1683
7  2011-07-01    1657
8  2011-08-01    1459
9  2011-09-01    1994
10 2011-10-01    2275
11 2011-11-01    3021
12 2011-12-01     869


In [13]:
fig_order_month = px.line(df_order_month, x='month', y='orders', title='Orders per Month')
fig_order_month.write_html("C:/Users/Decss/Desktop/SQL-övning/Analysis/HTML/monthly_orders.html",
    include_plotlyjs='cdn')
fig_order_month.show()

I det stora hela följer orderläggningen samma mönster som intäkterna. Den avviker marginellt vilket skulle kunna indikera att värde per order fluktuerar. Detta tittar vi vidare på nedan.

In [14]:
df_rev_order_m = db.sql("""
                      SELECT
                      DATE_TRUNC('month', invoicedate) as month,
                      SUM(total_price)/COUNT(DISTINCT invoiceno) as rev_per_order
                      FROM retail_enriched
                      WHERE total_price > 0
                      GROUP BY 1
                      ORDER BY 1
                      """).df()
print(df_rev_order_m)

        month  rev_per_order
0  2010-12-01     526.910026
1  2011-01-01     635.185645
2  2011-02-01     475.041418
3  2011-03-01     492.582710
4  2011-04-01     430.953845
5  2011-05-01     457.642243
6  2011-06-01     496.116771
7  2011-07-01     486.831268
8  2011-08-01     556.826877
9  2011-09-01     575.087203
10 2011-10-01     564.344966
11 2011-11-01     543.108263
12 2011-12-01     778.742772


In [15]:
fig_rev_order_m = px.line(df_rev_order_m, x='month', y='rev_per_order', title='Revenue per Order by Month')
fig_rev_order_m.write_html("C:/Users/Decss/Desktop/SQL-övning/Analysis/HTML/monthly_revenue_order.html",
    include_plotlyjs='cdn')
fig_rev_order_m.show()

När vi tittar på ordervärdet och antal ordrar så följer de i stort sett den underliggande trenden. Vi har däremot en månad som sticker ut och det är Januari 2011. Denna månad hade ett ovanligt högt värde per order.

Det skulle kunna vara värt att kika närmare på detta fenomen.
Kan det vara B2B kunder?
Kampanj?

# Sammanfattning


# Kvalitetsändring i data (uppd nedan)
* Initiala analys inkluderade alla positiva transaktioner. Även sådana som troligtvis inte hörde till försäljning.
    - En stor del av dessa inkluderas i "staging.top_anomalies". 


In [16]:
# Justerad data för veckovisa intäkter. Filtrerar bort poster som rimligen inte hör till försäljning.
df_rev_week_adj = db.sql("""
       SELECT
       DATE_TRUNC('week', invoicedate) as week_start,
       SUM(total_price) as revenue
       FROM retail_enriched
       WHERE total_price > 0 
       AND customerid IS NOT NULL
       AND description NOT ILIKE '%POSTAGE%'
       AND description NOT ILIKE '%AMAZON%'
       AND description NOT ILIKE '%Manual%'
       GROUP BY 1
       ORDER BY 1
       """).df()
print(df_rev_week_adj)  

   week_start     revenue
0  2010-11-29  147701.110
1  2010-12-06  210878.740
2  2010-12-13  161698.800
3  2010-12-20   45485.910
4  2011-01-03  113523.050
5  2011-01-10  152926.900
6  2011-01-17  174199.480
7  2011-01-24  103820.000
8  2011-01-31  104562.860
9  2011-02-07   87310.190
10 2011-02-14  125034.160
11 2011-02-21  129291.580
12 2011-02-28  115096.070
13 2011-03-07  108319.460
14 2011-03-14  136514.330
15 2011-03-21  127463.770
16 2011-03-28  142020.330
17 2011-04-04  109962.030
18 2011-04-11  133659.661
19 2011-04-18  114958.630
20 2011-04-25   72242.840
21 2011-05-02  122005.120
22 2011-05-09  176241.240
23 2011-05-16  194357.790
24 2011-05-23  142600.330
25 2011-05-30  100823.120
26 2011-06-06  192209.750
27 2011-06-13  170390.680
28 2011-06-20  110671.420
29 2011-06-27  115409.020
30 2011-07-04  129923.840
31 2011-07-11  116072.660
32 2011-07-18  165285.440
33 2011-07-25  162757.681
34 2011-08-01  154466.530
35 2011-08-08  139826.630
36 2011-08-15  166626.470
37 2011-08-2

In [17]:
# Plottar graf med justerad data
fig_rev_week_adj = px.line(
    df_rev_week_adj,
    x='week_start',
    y='revenue',
    title='Revenue per Week'
)
fig_rev_week_adj.write_html("C:/Users/Decss/Desktop/SQL-övning/Analysis/HTML/weekly_revenue_adj.html",
    include_plotlyjs='cdn')
fig_rev_week_adj.show()